In [1]:
import os
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import ta


In [2]:
DATA_FOLDER = "/home/hadoop/shareMarket_Data"

In [2]:
# ==========================================================
# Data Loader
# ==========================================================

class DataLoader:

    def __init__(

        self,

        report_file,

        data_folder

    ):

        self.report_file = report_file

        self.data_folder = data_folder

        print("Statistical DataLoader Initialized")
    
    # ======================================================
    # Load Bullish Shares
    # ======================================================

    def load_bullish_shares(self):

        """
        Reads Bullish_Shares.xlsx

        Returns
        -------
        list
        """

        if not os.path.exists(self.report_file):

            print("Bullish report not found.")

            return []

        df = pd.read_excel(self.report_file)

        if len(df) == 0:

            return []

        if "Symbol" not in df.columns:

            raise ValueError(
                "Column 'Symbol' not found."
            )

        symbols = (

            df["Symbol"]

            .dropna()

            .astype(str)

            .str.upper()

            .tolist()

        )

        return symbols

    # ======================================================
    # Load One 5 Minute File
    # ======================================================

    def load_5min_data(

        self,

        symbol

    ):

        filename = os.path.join(

            self.data_folder,

            f"{symbol}_5mins.txt"

        )

        if not os.path.exists(filename):

            print(f"{symbol} file not found.")

            return None

        df = pd.read_csv(filename)

        if len(df) == 0:

            return None

        df.columns = [

            c.strip()

            for c in df.columns

        ]

        numeric = [

            "Open",

            "High",

            "Low",

            "Close",

            "Volume"

        ]

        for col in numeric:

            df[col] = pd.to_numeric(

                df[col],

                errors="coerce"

            )

        df = df.dropna()

        df["Datetime"] = pd.to_datetime(

            df["date"].astype(str)

            + " "

            + df["time"].astype(str)

        )

        df = df.sort_values(

            "Datetime"

        ).reset_index(

            drop=True

        )

        return df

    # ======================================================
    # Load All 5 Minute Data
    # ======================================================

    def load_all(self):

        symbols = self.load_bullish_shares()

        all_data = {}

        for symbol in symbols:

            df = self.load_5min_data(symbol)

            if df is not None:

                all_data[symbol] = df

        return all_data

    # ======================================================
    # Refresh Every 5 Minutes
    # ======================================================

    def refresh(

        self,

        interval=300

    ):

        while True:

            print("="*80)

            print("Refreshing Data...")

            print("="*80)

            data = self.load_all()

            yield data

            time.sleep(interval)

Daily Data
         Symbol        Date   Time    Open    High     Low   Close     Volume  \
377  HINDUNILVR  2026-07-07  00:00  2200.0  2219.4  2193.7  2208.8  2552492.0   
378  HINDUNILVR  2026-07-08  00:00  2199.0  2199.0  2130.0  2135.8  1326689.0   
379  HINDUNILVR  2026-07-09  00:00  2139.0  2166.8  2137.1  2144.5  2201039.0   
380  HINDUNILVR  2026-07-10  00:00  2151.5  2178.0  2145.0  2150.6  1665179.0   
381  HINDUNILVR  2026-07-13  00:00  2143.0  2143.0  2120.0  2126.3   691157.0   

      DateTime  
377 2026-07-07  
378 2026-07-08  
379 2026-07-09  
380 2026-07-10  
381 2026-07-13  

5 Minute Data
           Symbol        Date   Time    Open    High     Low   Close   Volume  \
18681  HINDUNILVR  2026-07-13  12:25  2129.4  2129.4  2126.9  2128.0  13474.0   
18682  HINDUNILVR  2026-07-13  12:30  2128.0  2128.5  2126.0  2126.0  13198.0   
18683  HINDUNILVR  2026-07-13  12:35  2126.1  2132.0  2125.6  2132.0  45372.0   
18684  HINDUNILVR  2026-07-13  12:40  2131.9  2134.8  2130.5 

In [3]:
# ==========================================
# Module 2 : Indicator Engine
# File : indicators.py
# ==========================================

import pandas as pd
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange


class IndicatorEngine:

    def __init__(self):
        pass

    def calculate(self, df):

        df = df.copy()

        # -----------------------------
        # EMA
        # -----------------------------
        df["EMA20"] = EMAIndicator(
            close=df["Close"],
            window=20
        ).ema_indicator()

        df["EMA50"] = EMAIndicator(
            close=df["Close"],
            window=50
        ).ema_indicator()

        df["EMA200"] = EMAIndicator(
            close=df["Close"],
            window=200
        ).ema_indicator()

        # -----------------------------
        # RSI
        # -----------------------------
        df["RSI"] = RSIIndicator(
            close=df["Close"],
            window=14
        ).rsi()

        # -----------------------------
        # ATR
        # -----------------------------
        df["ATR"] = AverageTrueRange(
            high=df["High"],
            low=df["Low"],
            close=df["Close"],
            window=14
        ).average_true_range()

        # ATR Moving Average
        df["ATR_MA"] = (
            df["ATR"]
            .rolling(20)
            .mean()
        )

        # -----------------------------
        # MACD
        # -----------------------------
        macd = MACD(
            close=df["Close"],
            window_fast=12,
            window_slow=26,
            window_sign=9
        )

        df["MACD"] = macd.macd()
        df["MACD_SIGNAL"] = macd.macd_signal()
        df["MACD_HIST"] = macd.macd_diff()

        # -----------------------------
        # VWAP
        # -----------------------------
        typical_price = (
            df["High"] +
            df["Low"] +
            df["Close"]
        ) / 3

        cumulative_tpv = (
            typical_price *
            df["Volume"]
        ).cumsum()

        cumulative_volume = (
            df["Volume"]
        ).cumsum()

        df["VWAP"] = (
            cumulative_tpv /
            cumulative_volume
        )

        # -----------------------------
        # Average Volume
        # -----------------------------
        df["AVG_VOLUME20"] = (
            df["Volume"]
            .rolling(20)
            .mean()
        )

        # -----------------------------
        # Price Above VWAP
        # -----------------------------
        df["ABOVE_VWAP"] = (
            df["Close"] >
            df["VWAP"]
        )

        # -----------------------------
        # EMA Alignment
        # -----------------------------
        df["EMA_BULLISH"] = (
            (df["EMA20"] > df["EMA50"]) &
            (df["EMA50"] > df["EMA200"])
        )

        df["EMA_BEARISH"] = (
            (df["EMA20"] < df["EMA50"]) &
            (df["EMA50"] < df["EMA200"])
        )

        # -----------------------------
        # MACD Cross
        # -----------------------------
        df["MACD_BULLISH"] = (
            df["MACD"] >
            df["MACD_SIGNAL"]
        )

        df["MACD_BEARISH"] = (
            df["MACD"] <
            df["MACD_SIGNAL"]
        )

        # -----------------------------
        # ATR Increasing
        # -----------------------------
        df["ATR_INCREASING"] = (
            df["ATR"] >
            df["ATR_MA"]
        )

        # -----------------------------
        # High Volume
        # -----------------------------
        df["HIGH_VOLUME"] = (
            df["Volume"] >
            df["AVG_VOLUME20"]
        )

        return df

In [4]:
DATA_FOLDER = "/home/hadoop/shareMarket_Data"

loader = DataLoader(DATA_FOLDER)

daily_df, min5_df = loader.load_stock("HINDUNILVR")

engine = IndicatorEngine()

daily_df = engine.calculate(daily_df)
min5_df = engine.calculate(min5_df)

print(daily_df.tail(3))
print(min5_df.tail(3))

         Symbol        Date   Time    Open    High     Low   Close     Volume  \
379  HINDUNILVR  2026-07-09  00:00  2139.0  2166.8  2137.1  2144.5  2201039.0   
380  HINDUNILVR  2026-07-10  00:00  2151.5  2178.0  2145.0  2150.6  1665179.0   
381  HINDUNILVR  2026-07-13  00:00  2143.0  2143.0  2120.0  2126.3   691157.0   

      DateTime        EMA20  ...  MACD_HIST         VWAP  AVG_VOLUME20  \
379 2026-07-09  2173.129125  ...  -0.286920  2339.916079    1609396.50   
380 2026-07-10  2170.983494  ...  -1.402125  2339.452509    1633710.65   
381 2026-07-13  2166.727923  ...  -3.548673  2339.231122    1610460.85   

     ABOVE_VWAP  EMA_BULLISH  EMA_BEARISH  MACD_BULLISH  MACD_BEARISH  \
379       False        False         True         False          True   
380       False        False         True         False          True   
381       False        False         True         False          True   

     ATR_INCREASING  HIGH_VOLUME  
379           False         True  
380           F

In [5]:
# ==========================================
# Module 3 : Trend Engine
# ==========================================

class TrendEngine:

    def __init__(self):
        pass

    def detect_trend(self, df):

        latest = df.iloc[-1]

        score = 0

        reasons = []

        # ---------------------------------
        # EMA Trend
        # ---------------------------------

        if latest["EMA20"] > latest["EMA50"] > latest["EMA200"]:
            score += 30
            reasons.append("EMA Bullish")

        elif latest["EMA20"] < latest["EMA50"] < latest["EMA200"]:
            score -= 30
            reasons.append("EMA Bearish")

        # ---------------------------------
        # RSI
        # ---------------------------------

        if latest["RSI"] > 55:
            score += 15
            reasons.append("RSI Bullish")

        elif latest["RSI"] < 45:
            score -= 15
            reasons.append("RSI Bearish")

        # ---------------------------------
        # MACD
        # ---------------------------------

        if latest["MACD"] > latest["MACD_SIGNAL"]:
            score += 20
            reasons.append("MACD Bullish")

        else:
            score -= 20
            reasons.append("MACD Bearish")

        # ---------------------------------
        # VWAP
        # ---------------------------------

        if latest["ABOVE_VWAP"]:
            score += 10
            reasons.append("Above VWAP")

        else:
            score -= 10
            reasons.append("Below VWAP")

        # ---------------------------------
        # ATR
        # ---------------------------------

        if latest["ATR"] > latest["ATR_MA"]:
            score += 5
            reasons.append("ATR Increasing")

        # ---------------------------------
        # Price Position
        # ---------------------------------

        if latest["Close"] > latest["EMA20"]:
            score += 10
            reasons.append("Price Above EMA20")

        else:
            score -= 10
            reasons.append("Price Below EMA20")

        # ---------------------------------
        # Final Trend
        # ---------------------------------

        if score >= 40:
            trend = "Bullish"

        elif score <= -40:
            trend = "Bearish"

        else:
            trend = "Neutral"

        confidence = min(abs(score), 100)

        return {
            "Trend": trend,
            "Score": score,
            "Confidence": confidence,
            "Reasons": reasons
        }

In [6]:
trend_engine = TrendEngine()

trend = trend_engine.detect_trend(daily_df)

print("Trend :", trend["Trend"])
print("Score :", trend["Score"])
print("Confidence :", trend["Confidence"])

print()

for reason in trend["Reasons"]:
    print(reason)

Trend : Bearish
Score : -85
Confidence : 85

EMA Bearish
RSI Bearish
MACD Bearish
Below VWAP
Price Below EMA20


In [7]:
class SupportResistanceEngine:

    def __init__(
            self,
            swing_window=5,
            merge_percent=0.005,
            intraday_lookback=1000):
    
        self.swing_window = swing_window
        self.merge_percent = merge_percent
    
        # Only last N candles for 5-minute analysis
        self.intraday_lookback = intraday_lookback

    # =====================================================
    # Public Function
    # =====================================================

    def calculate(self,
                  daily_df,
                  intraday_df):

        daily = self._process_dataframe(
            daily_df,
            timeframe="Daily"
        )

        intraday = self._process_dataframe(
            intraday_df,
            timeframe="5 Minutes"
        )

        return {

            "current_price":
                intraday_df.iloc[-1]["Close"],

            "daily":
                daily,

            "intraday":
                intraday

        }

    
    
    
    

    # =====================================================
    # Swing High / Swing Low
    # =====================================================

    def _find_swings(self,
                     df):

        supports = []

        resistances = []

        w = self.swing_window

        for i in range(w,
                       len(df)-w):

            row = df.iloc[i]

            high = row["High"]

            low = row["Low"]

            left_high = df.iloc[
                i-w:i
            ]["High"].max()

            right_high = df.iloc[
                i+1:i+w+1
            ]["High"].max()

            left_low = df.iloc[
                i-w:i
            ]["Low"].min()

            right_low = df.iloc[
                i+1:i+w+1
            ]["Low"].min()

            # -----------------------

            if high > left_high and high > right_high:

                resistances.append({

                    "price": float(high),

                    "datetime":
                        row["DateTime"],

                    "volume":
                        float(row["Volume"])

                })

            # -----------------------

            if low < left_low and low < right_low:

                supports.append({

                    "price": float(low),

                    "datetime":
                        row["DateTime"],

                    "volume":
                        float(row["Volume"])

                })

        return {

            "supports": supports,

            "resistances": resistances

        }

    # =====================================================
    # Merge Nearby Levels
    # =====================================================

    def _merge_levels(self,
                      levels):

        if len(levels) == 0:
            return []

        levels = sorted(
            levels,
            key=lambda x: x["price"]
        )

        merged = []

        current = levels[0]

        current["touches"] = 1

        for level in levels[1:]:

            diff = abs(
                level["price"] -
                current["price"]
            )

            if diff/current["price"] <= self.merge_percent:

                current["price"] = round(

                    (
                        current["price"] +
                        level["price"]
                    ) / 2,

                    2

                )

                current["touches"] += 1

                current["volume"] += level["volume"]

                if level["datetime"] > current["datetime"]:

                    current["datetime"] = level["datetime"]

            else:

                merged.append(current)

                current = level

                current["touches"] = 1

        merged.append(current)

        return merged

    # =====================================================
    # Pivot Points
    # =====================================================

    def _pivot_points(self,
                      df):

        prev = df.iloc[-2]

        high = prev["High"]

        low = prev["Low"]

        close = prev["Close"]

        pp = (high+low+close)/3

        r1 = (2*pp)-low

        s1 = (2*pp)-high

        r2 = pp+(high-low)

        s2 = pp-(high-low)

        r3 = high+2*(pp-low)

        s3 = low-2*(high-pp)

        return {

            "PP": round(pp,2),

            "R1": round(r1,2),

            "R2": round(r2,2),

            "R3": round(r3,2),

            "S1": round(s1,2),

            "S2": round(s2,2),

            "S3": round(s3,2)

        }

    # =====================================================
    # EMA Levels
    # =====================================================

    def _ema_levels(self,
                    df):

        last = df.iloc[-1]

        return {

            "EMA20":
                round(last["EMA20"],2),

            "EMA50":
                round(last["EMA50"],2),

            "EMA200":
                round(last["EMA200"],2)

        }

    # =====================================================
    # Replace _process_dataframe()
    # =====================================================
    
    def _process_dataframe(self, df, timeframe):

        # ---------------------------------------------
        # Current Price
        # ---------------------------------------------
        current_price = float(df.iloc[-1]["Close"])
    
        # ---------------------------------------------
        # Swing Highs / Swing Lows
        # ---------------------------------------------
        swings = self._find_swings(df)
    
        # ---------------------------------------------
        # Merge Nearby Levels
        # ---------------------------------------------
        supports = self._merge_levels(
            swings["supports"]
        )
    
        resistances = self._merge_levels(
            swings["resistances"]
        )
    
        # ---------------------------------------------
        # ATR
        # ---------------------------------------------
        atr = float(df.iloc[-1]["ATR"])
    
        # ---------------------------------------------
        # Create Support Zones
        # ---------------------------------------------
        support_zones = self._create_zones(
            supports,
            atr,
            zone_type="Support"
        )
    
        # ---------------------------------------------
        # Create Resistance Zones
        # ---------------------------------------------
        resistance_zones = self._create_zones(
            resistances,
            atr,
            zone_type="Resistance"
        )
    
        # ---------------------------------------------
        # Strength Calculation
        # ---------------------------------------------
        support_zones = self._calculate_strength(
            df,
            support_zones,
            zone_type="Support"
        )
    
        resistance_zones = self._calculate_strength(
            df,
            resistance_zones,
            zone_type="Resistance"
        )
    
        # ---------------------------------------------
        # Keep Supports Below Current Price
        # ---------------------------------------------
        
        support_zones = [
            x for x in support_zones
            if x["price"] < current_price
        ]
        
        # ---------------------------------------------
        # Keep Resistances Above Current Price
        # ---------------------------------------------
        
        resistance_zones = [
            x for x in resistance_zones
            if x["price"] > current_price
        ]
        
        # ---------------------------------------------
        # Rank Levels
        # ---------------------------------------------
        
        support_zones = self._rank_levels(
            support_zones,
            current_price,
            df
        )
        
        resistance_zones = self._rank_levels(
            resistance_zones,
            current_price,
            df
        )
    
        # ---------------------------------------------
        # Nearest Levels
        # ---------------------------------------------
        nearest_support = self._nearest_support(
            support_zones,
            current_price
        )
    
        nearest_resistance = self._nearest_resistance(
            resistance_zones,
            current_price
        )
    
        # ---------------------------------------------
        # Pivot Points
        # ---------------------------------------------
        pivot_points = self._pivot_points(df)
    
        # ---------------------------------------------
        # EMA Levels
        # ---------------------------------------------
        ema_levels = self._ema_levels(df)
    
        # ---------------------------------------------
        # Return
        # ---------------------------------------------
        return {
    
            "timeframe": timeframe,
    
            "current_price": current_price,
    
            "supports": support_zones,
    
            "resistances": resistance_zones,
    
            "nearest_support": nearest_support,
    
            "nearest_resistance": nearest_resistance,
    
            "pivot_points": pivot_points,
    
            "ema_levels": ema_levels,
    
            "support_count": len(support_zones),
    
            "resistance_count": len(resistance_zones)
    
        }
    
    
    # =====================================================
    # Create Zones
    # =====================================================
    
    def _create_zones(
            self,
            levels,
            atr,
            zone_type):
    
        zones = []
    
        width = atr * 0.50
    
        for level in levels:
    
            price = level["price"]
    
            zones.append({
    
                "type": zone_type,
    
                "price": round(price,2),
    
                "zone_low": round(price-width,2),
    
                "zone_high": round(price+width,2),
    
                "touches": level["touches"],
    
                "volume": level["volume"],
    
                "datetime": level["datetime"]
    
            })
    
        return zones
    
    
    # =====================================================
    # Count Zone Touches
    # =====================================================
    
    def _count_zone_touches(
            self,
            df,
            zone):
    
        touches = 0
    
        for _, row in df.iterrows():
    
            if row["Low"] <= zone["zone_high"] and \
               row["High"] >= zone["zone_low"]:
    
                touches += 1
    
        return touches
    
    
    # =====================================================
    # Calculate Strength
    # =====================================================
    
    def _calculate_strength(
            self,
            df,
            zones,
            zone_type):
    
        avg_volume = df["Volume"].mean()
    
        current_price = float(df.iloc[-1]["Close"])
    
        latest_date = df.iloc[-1]["DateTime"]
    
        result = []
    
        for zone in zones:
    
            score = 0
    
            # -----------------------------
            # Historical touches
            # -----------------------------
    
            historical = self._count_zone_touches(
                df,
                zone
            )
    
            total_touches = max(
                zone["touches"],
                historical
            )
    
            zone["touches"] = total_touches
    
            score += min(
                total_touches * 5,
                35
            )
    
            # -----------------------------
            # Volume
            # -----------------------------
    
            if zone["volume"] >= avg_volume:
    
                score += 20
    
            else:
    
                score += 10
    
            # -----------------------------
            # Distance
            # -----------------------------
    
            distance = abs(
                current_price -
                zone["price"]
            )
    
            zone["distance"] = round(
                distance,
                2
            )
    
            score += max(
                0,
                30 -
                (
                    distance/current_price
                )*100
            )
    
            # -----------------------------
            # Recency
            # -----------------------------
    
            days = (
                latest_date -
                zone["datetime"]
            ).days
    
            score += max(
                0,
                15-days
            )
    
            # -----------------------------
            # Final Strength
            # -----------------------------
    
            zone["strength"] = round(
                min(score,100),
                2
            )
    
            result.append(zone)
    
        return result

       
    # =====================================================
    # Replace calculate()
    # =====================================================
    
    def calculate(
            self,
            daily_df,
            intraday_df):
    
        daily = self._process_dataframe(
            daily_df,
            "Daily"
        )
    
        intraday = self._process_dataframe(
            intraday_df,
            "5 Minutes"
        )
    
        breakout = self._detect_breakout(
            intraday_df,
            intraday["resistances"],
            intraday["supports"]
        )
    
        retest = self._detect_retest(
            intraday_df,
            breakout
        )
    
        return {
    
            "current_price": float(
                intraday_df.iloc[-1]["Close"]
            ),
    
            "daily": daily,
    
            "intraday": intraday,
    
            "breakout": breakout,
    
            "retest": retest
    
        }
        
        
    # =====================================================
    # Detect Breakout
    # =====================================================
    
    def _detect_breakout(
            self,
            df,
            resistances,
            supports):
    
        if len(df) < 2:
    
            return None
    
        last = df.iloc[-1]
    
        previous = df.iloc[-2]
    
        close = float(last["Close"])
    
        volume = float(last["Volume"])
    
        avg_volume = float(df["Volume"].tail(20).mean())
    
        # -----------------------------
        # Resistance Breakout
        # -----------------------------
    
        for level in resistances:
    
            if previous["Close"] <= level["price"] \
               and close > level["price"] \
               and volume > avg_volume:
    
                return {
    
                    "direction": "Bullish",
    
                    "level": level["price"],
    
                    "datetime": last["DateTime"],
    
                    "close": close,
    
                    "volume": volume,
    
                    "strength": level["strength"]
    
                }
    
        # -----------------------------
        # Support Breakdown
        # -----------------------------
    
        for level in supports:
    
            if previous["Close"] >= level["price"] \
               and close < level["price"] \
               and volume > avg_volume:
    
                return {
    
                    "direction": "Bearish",
    
                    "level": level["price"],
    
                    "datetime": last["DateTime"],
    
                    "close": close,
    
                    "volume": volume,
    
                    "strength": level["strength"]
    
                }
    
        return None
        
        
    # =====================================================
    # Detect Retest
    # =====================================================
    
    def _detect_retest(
            self,
            df,
            breakout):
    
        if breakout is None:
    
            return None
    
        level = breakout["level"]
    
        direction = breakout["direction"]
    
        recent = df.tail(10)
    
        for _, row in recent.iterrows():
    
            high = float(row["High"])
    
            low = float(row["Low"])
    
            close = float(row["Close"])
    
            if direction == "Bullish":
    
                if low <= level <= high and close > level:
    
                    return {
    
                        "status": True,
    
                        "direction": "Bullish",
    
                        "level": level,
    
                        "datetime": row["DateTime"]
    
                    }
    
            if direction == "Bearish":
    
                if low <= level <= high and close < level:
    
                    return {
    
                        "status": True,
    
                        "direction": "Bearish",
    
                        "level": level,
    
                        "datetime": row["DateTime"]
    
                    }
    
        return {
    
            "status": False,
    
            "direction": direction,
    
            "level": level
    
        }
    
        
    # =====================================================
    # Nearest Support
    # =====================================================
    
    def _nearest_support(
            self,
            levels,
            current_price):
    
        levels = [
    
            x for x in levels
    
            if x["price"] < current_price
    
        ]
    
        if len(levels) == 0:
    
            return None
    
        return levels[0]
    
    
    # =====================================================
    # Nearest Resistance
    # =====================================================
    
    def _nearest_resistance(
            self,
            levels,
            current_price):
    
        levels = [
    
            x for x in levels
    
            if x["price"] > current_price
    
        ]
    
        if len(levels) == 0:
    
            return None
    
        return levels[0]

    # =====================================================
    # _rank_levels Function
    # =====================================================
    def _rank_levels(self, levels, current_price, df):
    
    
        if len(levels) == 0:
            return []
    
        latest = df.iloc[-1]["DateTime"]
    
        ranked = []
    
        for level in levels:
    
            distance = abs(current_price - level["price"])
            distance_score = max(0, 100 - distance)
    
            strength_score = level["strength"]
    
            age = (latest - level["datetime"]).days
            recency_score = max(0, 100 - age)
    
            final_score = (
                distance_score * 0.50 +
                strength_score * 0.35 +
                recency_score * 0.15
            )
    
            level["rank_score"] = round(final_score, 2)
    
            ranked.append(level)
    
        ranked.sort(
            key=lambda x: x["rank_score"],
            reverse=True
        )
    
        return ranked
    
    # =====================================================
    # Final calculate()
    # =====================================================
    
    def calculate(
            self,
            daily_df,
            intraday_df):
    
        # -------------------------------
        # Process Daily
        # -------------------------------
    
        daily = self._process_dataframe(
            daily_df,
            "Daily"
        )
    
        # -------------------------------
        # Process Intraday
        # -------------------------------
    
        intraday = self._process_dataframe(
            intraday_df,
            "5 Minutes"
        )
    
        # -------------------------------
        # Current Price
        # -------------------------------
    
        current_price = float(
            intraday_df.iloc[-1]["Close"]
        )
    
        # -------------------------------
        # Breakout
        # -------------------------------
    
        breakout = self._detect_breakout(
            intraday_df,
            intraday["resistances"],
            intraday["supports"]
        )
    
        # -------------------------------
        # Retest
        # -------------------------------
    
        retest = self._detect_retest(
            intraday_df,
            breakout
        )
    
        # -------------------------------
        # Rank Daily Levels
        # -------------------------------
    
        daily_supports = sorted(
            daily["supports"],
            key=lambda x: (
                x["strength"],
                x["touches"]
            ),
            reverse=True
        )
    
        daily_resistances = sorted(
            daily["resistances"],
            key=lambda x: (
                x["strength"],
                x["touches"]
            ),
            reverse=True
        )
    
        # -------------------------------
        # Rank Intraday Levels
        # -------------------------------
    
        intraday_supports = sorted(
            intraday["supports"],
            key=lambda x: (
                x["strength"],
                x["touches"]
            ),
            reverse=True
        )
    
        intraday_resistances = sorted(
            intraday["resistances"],
            key=lambda x: (
                x["strength"],
                x["touches"]
            ),
            reverse=True
        )
    
        # -------------------------------
        # Nearest Levels
        # -------------------------------
    
        daily_nearest_support = self._nearest_support(
            daily_supports,
            current_price
        )
    
        daily_nearest_resistance = self._nearest_resistance(
            daily_resistances,
            current_price
        )
    
        intraday_nearest_support = self._nearest_support(
            intraday_supports,
            current_price
        )
    
        intraday_nearest_resistance = self._nearest_resistance(
            intraday_resistances,
            current_price
        )
    
        # -------------------------------
        # Return
        # -------------------------------
    
        return {
    
            "symbol": str(
                intraday_df.iloc[-1]["Symbol"]
            ),
    
            "current_price": current_price,
    
            "daily": {
    
                "supports": daily_supports,
    
                "resistances": daily_resistances,
    
                "nearest_support":
                    daily_nearest_support,
    
                "nearest_resistance":
                    daily_nearest_resistance,
    
                "pivot_points":
                    daily["pivot_points"],
    
                "ema_levels":
                    daily["ema_levels"]
    
            },
    
            "intraday": {
    
                "supports":
                    intraday_supports,
    
                "resistances":
                    intraday_resistances,
    
                "nearest_support":
                    intraday_nearest_support,
    
                "nearest_resistance":
                    intraday_nearest_resistance,
    
                "pivot_points":
                    intraday["pivot_points"],
    
                "ema_levels":
                    intraday["ema_levels"]
    
            },
    
            "breakout":
                breakout,
    
            "retest":
                retest
    
        }

In [8]:
# ==========================================================
# RUN SUPPORT & RESISTANCE ENGINE (UPDATED)
# ==========================================================

DATA_FOLDER = "/home/hadoop/shareMarket_Data"
SYMBOL = "HINDUNILVR"

# ----------------------------------------------------------
# Load Data
# ----------------------------------------------------------

loader = DataLoader(DATA_FOLDER)

daily_df, min5_df = loader.load_stock(SYMBOL)

# ----------------------------------------------------------
# Calculate Indicators
# ----------------------------------------------------------

indicator = IndicatorEngine()

daily_df = indicator.calculate(daily_df)
min5_df = indicator.calculate(min5_df)

# ----------------------------------------------------------
# Run Support & Resistance Engine
# ----------------------------------------------------------

sr = SupportResistanceEngine(
    swing_window=5,
    merge_percent=0.005,
    intraday_lookback=1000
)

levels = sr.calculate(
    daily_df=daily_df,
    intraday_df=min5_df
)

# ==========================================================
# SUMMARY
# ==========================================================

print("="*100)
print(f"SYMBOL         : {SYMBOL}")
print(f"CURRENT PRICE  : {levels['current_price']}")
print("="*100)

# ==========================================================
# DAILY
# ==========================================================

print("\nDAILY SUPPORT")
print("-"*100)
print(levels["daily"]["nearest_support"])

print("\nDAILY RESISTANCE")
print("-"*100)
print(levels["daily"]["nearest_resistance"])

print("\nDAILY PIVOT")
print("-"*100)

for k, v in levels["daily"]["pivot_points"].items():
    print(f"{k:<5}: {v}")

print("\nDAILY EMA")
print("-"*100)

for k, v in levels["daily"]["ema_levels"].items():
    print(f"{k:<8}: {v}")

# ==========================================================
# 5 MINUTE
# ==========================================================

print("\n5 MIN SUPPORT")
print("-"*100)
print(levels["intraday"]["nearest_support"])

print("\n5 MIN RESISTANCE")
print("-"*100)
print(levels["intraday"]["nearest_resistance"])

print("\n5 MIN PIVOT")
print("-"*100)

for k, v in levels["intraday"]["pivot_points"].items():
    print(f"{k:<5}: {v}")

print("\n5 MIN EMA")
print("-"*100)

for k, v in levels["intraday"]["ema_levels"].items():
    print(f"{k:<8}: {v}")

# ==========================================================
# BREAKOUT
# ==========================================================

print("\nBREAKOUT")
print("-"*100)

if levels["breakout"] is None:
    print("No Breakout")
else:
    print(levels["breakout"])

# ==========================================================
# RETEST
# ==========================================================

print("\nRETEST")
print("-"*100)

if levels["retest"] is None:
    print("No Retest")
else:
    print(levels["retest"])

# ==========================================================
# TOP DAILY SUPPORTS
# ==========================================================

print("\nTOP DAILY SUPPORTS")
print("="*100)

for i, level in enumerate(levels["daily"]["supports"], start=1):

    print(f"\nSupport {i}")
    print(f"Price      : {level['price']}")
    print(f"Zone       : {level['zone_low']} - {level['zone_high']}")
    print(f"Strength   : {level['strength']}")
    print(f"Rank Score : {level['rank_score']}")
    print(f"Touches    : {level['touches']}")
    print(f"Distance   : {level['distance']}")

# ==========================================================
# TOP DAILY RESISTANCES
# ==========================================================

print("\nTOP DAILY RESISTANCES")
print("="*100)

for i, level in enumerate(levels["daily"]["resistances"], start=1):

    print(f"\nResistance {i}")
    print(f"Price      : {level['price']}")
    print(f"Zone       : {level['zone_low']} - {level['zone_high']}")
    print(f"Strength   : {level['strength']}")
    print(f"Rank Score : {level['rank_score']}")
    print(f"Touches    : {level['touches']}")
    print(f"Distance   : {level['distance']}")

# ==========================================================
# TOP 5-MIN SUPPORTS
# ==========================================================

print("\nTOP 5-MIN SUPPORTS")
print("="*100)

if len(levels["intraday"]["supports"]) == 0:
    print("No Support Found")

for i, level in enumerate(levels["intraday"]["supports"], start=1):

    print(f"\nSupport {i}")
    print(f"Price      : {level['price']}")
    print(f"Zone       : {level['zone_low']} - {level['zone_high']}")
    print(f"Strength   : {level['strength']}")
    print(f"Rank Score : {level['rank_score']}")
    print(f"Touches    : {level['touches']}")
    print(f"Distance   : {level['distance']}")

# ==========================================================
# TOP 5-MIN RESISTANCES
# ==========================================================

print("\nTOP 5-MIN RESISTANCES")
print("="*100)

if len(levels["intraday"]["resistances"]) == 0:
    print("No Resistance Found")

for i, level in enumerate(levels["intraday"]["resistances"], start=1):

    print(f"\nResistance {i}")
    print(f"Price      : {level['price']}")
    print(f"Zone       : {level['zone_low']} - {level['zone_high']}")
    print(f"Strength   : {level['strength']}")
    print(f"Rank Score : {level['rank_score']}")
    print(f"Touches    : {level['touches']}")
    print(f"Distance   : {level['distance']}")

print("\n")
print("="*100)
print("SUPPORT & RESISTANCE ENGINE COMPLETED")
print("="*100)

SYMBOL         : HINDUNILVR
CURRENT PRICE  : 2134.3

DAILY SUPPORT
----------------------------------------------------------------------------------------------------
{'type': 'Support', 'price': 2114.2, 'zone_low': 2093.04, 'zone_high': 2135.36, 'touches': 31, 'volume': 2364422.0, 'datetime': Timestamp('2026-06-30 00:00:00'), 'distance': 12.1, 'strength': 86.43, 'rank_score': 87.25}

DAILY RESISTANCE
----------------------------------------------------------------------------------------------------
{'type': 'Resistance', 'price': 2192.8, 'zone_low': 2171.64, 'zone_high': 2213.96, 'touches': 52, 'volume': 3794437.0, 'datetime': Timestamp('2026-04-08 00:00:00'), 'distance': 66.5, 'strength': 81.87, 'rank_score': 46.0}

DAILY PIVOT
----------------------------------------------------------------------------------------------------
PP   : 2157.87
R1   : 2170.73
R2   : 2190.87
R3   : 2203.73
S1   : 2137.73
S2   : 2124.87
S3   : 2104.73

DAILY EMA
-----------------------------------------

Class Chart pattern - Done

In [9]:
# ==========================================================
# Chart Pattern Engine Class
# Detect:
# - Double Bottom
# - Double Top
# - Triple Bottom
# - Triple Top
# - Head & Shoulders
# - Inverse Head & Shoulders
# - Triangle
# - Rising Triangle
# - Falling Triangle
# - Wedge
# - Flag / Pennant
# - Cup & Handle
# ==========================================================


class ChartPatternEngine:

    def __init__(
            self,
            swing_window=5,
            tolerance=0.03,
            lookback=100):

        self.swing_window = swing_window
        self.tolerance = tolerance
        self.lookback = lookback


    # ======================================================
    # Swing Detection
    # ======================================================

    def _find_swings(self, df):

        highs = []
        lows = []

        for i in range(
            self.swing_window,
            len(df)-self.swing_window
        ):

            high = df["High"].iloc[i]
            low = df["Low"].iloc[i]

            prev_highs = df["High"].iloc[
                i-self.swing_window:i
            ]

            next_highs = df["High"].iloc[
                i+1:i+self.swing_window+1
            ]

            prev_lows = df["Low"].iloc[
                i-self.swing_window:i
            ]

            next_lows = df["Low"].iloc[
                i+1:i+self.swing_window+1
            ]


            if (
                high >= prev_highs.max()
                and high >= next_highs.max()
            ):
                highs.append(
                    {
                        "index": i,
                        "price": high,
                        "date": df.iloc[i]["DateTime"]
                    }
                )


            if (
                low <= prev_lows.min()
                and low <= next_lows.min()
            ):
                lows.append(
                    {
                        "index": i,
                        "price": low,
                        "date": df.iloc[i]["DateTime"]
                    }
                )


        return highs, lows


    # ======================================================
    # Double Bottom
    # ======================================================

    def _double_bottom(
            self,
            lows):

        if len(lows) < 2:
            return None


        first = lows[-2]
        second = lows[-1]


        diff = abs(
            first["price"]
            -
            second["price"]
        ) / first["price"]


        if diff <= self.tolerance:

            return {

                "pattern":
                    "Double Bottom",

                "direction":
                    "Bullish",

                "confidence":
                    round(
                        (1-diff)*100,
                        2
                    ),

                "levels":[
                    first["price"],
                    second["price"]
                ]
            }


        return None


    # ======================================================
    # Double Top
    # ======================================================

    def _double_top(
            self,
            highs):

        if len(highs) < 2:
            return None


        first = highs[-2]
        second = highs[-1]


        diff = abs(
            first["price"]
            -
            second["price"]
        ) / first["price"]


        if diff <= self.tolerance:

            return {

                "pattern":
                    "Double Top",

                "direction":
                    "Bearish",

                "confidence":
                    round(
                        (1-diff)*100,
                        2
                    ),

                "levels":[
                    first["price"],
                    second["price"]
                ]
            }


        return None



    # ======================================================
    # Head & Shoulders
    # ======================================================

    def _head_shoulders(
            self,
            highs):

        if len(highs) < 3:
            return None


        left = highs[-3]["price"]
        head = highs[-2]["price"]
        right = highs[-1]["price"]


        if (
            head > left
            and
            head > right
        ):

            return {

                "pattern":
                    "Head & Shoulders",

                "direction":
                    "Bearish",

                "confidence":
                    75

            }


        return None



    # ======================================================
    # Inverse Head & Shoulders
    # ======================================================

    def _inverse_head_shoulders(
            self,
            lows):

        if len(lows) < 3:
            return None


        left = lows[-3]["price"]
        head = lows[-2]["price"]
        right = lows[-1]["price"]


        if (
            head < left
            and
            head < right
        ):

            return {

                "pattern":
                    "Inverse Head & Shoulders",

                "direction":
                    "Bullish",

                "confidence":
                    75

            }


        return None



    # ======================================================
    # Triangle Detection
    # ======================================================

    def _triangle(
            self,
            df):

        recent = df.tail(
            self.lookback
        )


        high_slope = np.polyfit(
            range(len(recent)),
            recent["High"],
            1
        )[0]


        low_slope = np.polyfit(
            range(len(recent)),
            recent["Low"],
            1
        )[0]


        if (
            high_slope < 0
            and
            low_slope > 0
        ):

            return {

                "pattern":
                    "Symmetrical Triangle",

                "direction":
                    "Neutral",

                "confidence":
                    70
            }


        if (
            abs(high_slope) < 0.01
            and
            low_slope > 0
        ):

            return {

                "pattern":
                    "Ascending Triangle",

                "direction":
                    "Bullish",

                "confidence":
                    70
            }


        if (
            high_slope < 0
            and
            abs(low_slope) < 0.01
        ):

            return {

                "pattern":
                    "Descending Triangle",

                "direction":
                    "Bearish",

                "confidence":
                    70
            }


        return None



    # ======================================================
    # Wedge
    # ======================================================

    def _wedge(self, df):

        recent = df.tail(
            self.lookback
        )


        high_slope = np.polyfit(
            range(len(recent)),
            recent["High"],
            1
        )[0]


        low_slope = np.polyfit(
            range(len(recent)),
            recent["Low"],
            1
        )[0]


        if (
            high_slope > 0
            and
            low_slope > 0
            and
            high_slope > low_slope
        ):

            return {

                "pattern":
                    "Rising Wedge",

                "direction":
                    "Bearish",

                "confidence":
                    65
            }


        if (
            high_slope < 0
            and
            low_slope < 0
            and
            low_slope < high_slope
        ):

            return {

                "pattern":
                    "Falling Wedge",

                "direction":
                    "Bullish",

                "confidence":
                    65
            }


        return None



    # ======================================================
    # Main Calculate Function
    # ======================================================

    def calculate(
            self,
            daily_df):


        df = daily_df.tail(
            self.lookback
        ).copy()


        highs, lows = self._find_swings(df)


        patterns = []


        detectors = [

            self._double_bottom(lows),

            self._double_top(highs),

            self._head_shoulders(highs),

            self._inverse_head_shoulders(lows),

            self._triangle(df),

            self._wedge(df)

        ]


        for pattern in detectors:

            if pattern:

                patterns.append(pattern)


        if len(patterns) == 0:

            return {

                "pattern":
                    "No Pattern",

                "direction":
                    "Neutral",

                "patterns": []

            }


        patterns = sorted(
            patterns,
            key=lambda x:
            x["confidence"],
            reverse=True
        )


        return {

            "pattern":
                patterns[0]["pattern"],

            "direction":
                patterns[0]["direction"],

            "confidence":
                patterns[0]["confidence"],

            "patterns":
                patterns

        }

In [10]:
# ==========================================================
# RUN CHART PATTERN ENGINE
# ==========================================================

pattern_engine = ChartPatternEngine()

pattern_result = pattern_engine.calculate(
    daily_df
)

print("="*80)
print("CHART PATTERN RESULT")
print("="*80)

print(pattern_result)

CHART PATTERN RESULT
{'pattern': 'Double Top', 'direction': 'Bearish', 'confidence': np.float64(99.34), 'patterns': [{'pattern': 'Double Top', 'direction': 'Bearish', 'confidence': np.float64(99.34), 'levels': [np.float64(2224.6), np.float64(2239.3)]}, {'pattern': 'Double Bottom', 'direction': 'Bullish', 'confidence': np.float64(97.65), 'levels': [np.float64(2065.7), np.float64(2114.2)]}, {'pattern': 'Inverse Head & Shoulders', 'direction': 'Bullish', 'confidence': 75}]}


**Price Action Engine - Done**

In [11]:
# ==========================================================
# Price Action Engine
# 5 Minute Entry Logic
#
# Detect:
# - Bullish / Bearish Candles
# - Engulfing Pattern
# - Hammer
# - Shooting Star
# - Higher High / Higher Low
# - Lower High / Lower Low
# - Break of Structure (BOS)
# - Change of Character (CHOCH)
# - Pullback / Retest
# - Entry Signal
# ==========================================================

import pandas as pd
import numpy as np


class PriceActionEngine:


    def __init__(
            self,
            lookback=50,
            tolerance=0.002):

        self.lookback = lookback
        self.tolerance = tolerance



    # ======================================================
    # Candle Analysis
    # ======================================================

    def _candle_data(self, row):

        body = abs(
            row["Close"] -
            row["Open"]
        )

        upper_wick = (
            row["High"]
            -
            max(row["Close"], row["Open"])
        )

        lower_wick = (
            min(row["Close"], row["Open"])
            -
            row["Low"]
        )

        return {

            "body": body,

            "upper_wick": upper_wick,

            "lower_wick": lower_wick

        }



    # ======================================================
    # Bullish Engulfing
    # ======================================================

    def _bullish_engulfing(self, df):

        prev = df.iloc[-2]
        curr = df.iloc[-1]


        return (

            prev["Close"] < prev["Open"]

            and

            curr["Close"] > curr["Open"]

            and

            curr["Open"] < prev["Close"]

            and

            curr["Close"] > prev["Open"]

        )



    # ======================================================
    # Bearish Engulfing
    # ======================================================

    def _bearish_engulfing(self, df):

        prev = df.iloc[-2]
        curr = df.iloc[-1]


        return (

            prev["Close"] > prev["Open"]

            and

            curr["Close"] < curr["Open"]

            and

            curr["Open"] > prev["Close"]

            and

            curr["Close"] < prev["Open"]

        )



    # ======================================================
    # Hammer
    # ======================================================

    def _hammer(self, df):

        row = df.iloc[-1]

        c = self._candle_data(row)


        return (

            c["lower_wick"] >
            c["body"] * 2

            and

            c["upper_wick"] <
            c["body"]

        )



    # ======================================================
    # Shooting Star
    # ======================================================

    def _shooting_star(self, df):

        row = df.iloc[-1]

        c = self._candle_data(row)


        return (

            c["upper_wick"] >
            c["body"] * 2

            and

            c["lower_wick"] <
            c["body"]

        )



    # ======================================================
    # Market Structure
    # ======================================================

    def _market_structure(self, df):

        recent = df.tail(10)


        highs = recent["High"].values
        lows = recent["Low"].values


        if (
            highs[-1] > highs[-3]
            and
            lows[-1] > lows[-3]
        ):

            return "Bullish"



        if (
            highs[-1] < highs[-3]
            and
            lows[-1] < lows[-3]
        ):

            return "Bearish"


        return "Neutral"



    # ======================================================
    # Break Of Structure
    # ======================================================

    def _detect_bos(self, df):

        recent = df.tail(
            self.lookback
        )


        current = recent.iloc[-1]


        previous_high = (
            recent["High"]
            .iloc[:-1]
            .max()
        )


        previous_low = (
            recent["Low"]
            .iloc[:-1]
            .min()
        )


        if current["Close"] > previous_high:

            return {

                "signal":
                    "Bullish BOS",

                "level":
                    previous_high

            }



        if current["Close"] < previous_low:

            return {

                "signal":
                    "Bearish BOS",

                "level":
                    previous_low

            }



        return None



    # ======================================================
    # Change Of Character
    # ======================================================

    def _detect_choch(self, df):

        structure = self._market_structure(df)


        bos = self._detect_bos(df)


        if bos is None:

            return None


        if (
            structure == "Bearish"
            and
            bos["signal"] == "Bullish BOS"
        ):

            return "Bullish CHOCH"



        if (
            structure == "Bullish"
            and
            bos["signal"] == "Bearish BOS"
        ):

            return "Bearish CHOCH"


        return None



    # ======================================================
    # Retest Detection
    # ======================================================

    def _detect_retest(
            self,
            df,
            level):


        if level is None:

            return False


        current = df.iloc[-1]


        return (

            current["Low"] <= level

            and

            current["Close"] > level

        )



    # ======================================================
    # Main Calculation
    # ======================================================

    def calculate(
            self,
            df,
            support=None,
            resistance=None):


        df = df.tail(
            self.lookback
        ).copy()


        current = df.iloc[-1]


        patterns = []


        if self._bullish_engulfing(df):

            patterns.append(
                "Bullish Engulfing"
            )


        if self._bearish_engulfing(df):

            patterns.append(
                "Bearish Engulfing"
            )


        if self._hammer(df):

            patterns.append(
                "Hammer"
            )


        if self._shooting_star(df):

            patterns.append(
                "Shooting Star"
            )


        structure = self._market_structure(df)


        bos = self._detect_bos(df)

        choch = self._detect_choch(df)



        # -----------------------------------
        # Retest
        # -----------------------------------

        retest = False

        if support:

            retest = self._detect_retest(
                df,
                support["price"]
            )


        # -----------------------------------
        # Score
        # -----------------------------------

        score = 0


        if structure == "Bullish":

            score += 30


        if "Bullish Engulfing" in patterns:

            score += 20


        if "Hammer" in patterns:

            score += 15


        if bos and "Bullish" in bos["signal"]:

            score += 20


        if retest:

            score += 15



        # -----------------------------------
        # Signal
        # -----------------------------------

        if score >= 60:

            signal = "LONG"


        elif score <= 30:

            signal = "SHORT"


        else:

            signal = "NO TRADE"



        return {

            "signal":
                signal,

            "score":
                score,

            "patterns":
                patterns,

            "market_structure":
                structure,

            "BOS":
                bos,

            "CHOCH":
                choch,

            "retest":
                retest,

            "current_price":
                current["Close"]

        }

In [12]:
# ==========================================================
# RUN PRICE ACTION ENGINE
# ==========================================================

pa_engine = PriceActionEngine()

price_action_result = pa_engine.calculate(
    min5_df,
    support=levels["daily"]["nearest_support"],
    resistance=levels["daily"]["nearest_resistance"]
)


print("="*80)
print("PRICE ACTION RESULT")
print("="*80)

for key, value in price_action_result.items():
    print(f"{key}: {value}")

PRICE ACTION RESULT
signal: SHORT
score: 30
patterns: ['Shooting Star']
market_structure: Bullish
BOS: None
CHOCH: None
retest: False
current_price: 2134.3


**Probability Engine - Done**

In [13]:
# ==========================================================
# Probability Engine
#
# Calculates trade winning probability based on:
#
# 1. Daily Trend
# 2. Chart Pattern
# 3. Support/Resistance Position
# 4. Price Action
# 5. RSI
# 6. EMA
# 7. MACD
# 8. VWAP
# 9. ATR Risk
#
# Output:
# - Probability %
# - Confidence
# - Trade Quality
# ==========================================================


class ProbabilityEngine:


    def __init__(self):

        pass



    # ======================================================
    # Trend Score
    # ======================================================

    def _trend_score(
            self,
            trend):

        if trend == "Bullish":

            return 20


        elif trend == "Bearish":

            return 0


        return 10



    # ======================================================
    # Chart Pattern Score
    # ======================================================

    def _pattern_score(
            self,
            pattern):

        if not pattern:

            return 0


        direction = pattern.get(
            "direction",
            "Neutral"
        )


        confidence = pattern.get(
            "confidence",
            0
        )


        if direction == "Bullish":

            return min(
                15,
                confidence * 0.15
            )


        return 0



    # ======================================================
    # Support Resistance Score
    # ======================================================

    def _sr_score(
            self,
            price,
            support,
            resistance):


        score = 0


        if support:

            distance = abs(
                price -
                support["price"]
            )


            if distance <= (
                price * 0.02
            ):

                score += 10



        if resistance:

            distance = abs(
                resistance["price"]
                -
                price
            )


            if distance > (
                price * 0.03
            ):

                score += 5



        return score



    # ======================================================
    # Price Action Score
    # ======================================================

    def _price_action_score(
            self,
            price_action):


        score = 0


        if price_action["signal"] == "LONG":

            score += 20



        if price_action["BOS"]:

            if (
                "Bullish"
                in
                price_action["BOS"]["signal"]
            ):

                score += 5



        if price_action["retest"]:

            score += 5



        return score



    # ======================================================
    # Indicator Score
    # ======================================================

    def _indicator_score(
            self,
            row):

        score = 0


        # RSI

        if row["RSI"] > 50:

            score += 5



        # EMA

        if (
            row["EMA20"]
            >
            row["EMA50"]
        ):

            score += 5



        # MACD

        if (
            row["MACD"]
            >
            row["MACD_SIGNAL"]
        ):

            score += 5



        # VWAP

        if (
            row["Close"]
            >
            row["VWAP"]
        ):

            score += 5



        return score



    # ======================================================
    # Risk Score
    # ======================================================

    def _risk_score(
            self,
            entry,
            stop_loss,
            target):


        risk = abs(
            entry -
            stop_loss
        )


        reward = abs(
            target -
            entry
        )


        if risk == 0:

            return 0


        rr = reward / risk


        if rr >= 2:

            return 10


        elif rr >= 1.5:

            return 5


        return 0



    # ======================================================
    # Main Calculation
    # ======================================================

    def calculate(
            self,
            trend,
            pattern,
            price_action,
            current_price,
            daily_support,
            daily_resistance,
            indicator_row,
            stop_loss,
            target):


        score = 0


        # -----------------------------
        # Trend
        # -----------------------------

        score += self._trend_score(
            trend
        )


        # -----------------------------
        # Chart Pattern
        # -----------------------------

        score += self._pattern_score(
            pattern
        )


        # -----------------------------
        # S/R
        # -----------------------------

        score += self._sr_score(
            current_price,
            daily_support,
            daily_resistance
        )


        # -----------------------------
        # Price Action
        # -----------------------------

        score += self._price_action_score(
            price_action
        )


        # -----------------------------
        # Indicators
        # -----------------------------

        score += self._indicator_score(
            indicator_row
        )


        # -----------------------------
        # Risk Reward
        # -----------------------------

        score += self._risk_score(
            current_price,
            stop_loss,
            target
        )



        probability = min(
            100,
            round(score,2)
        )



        if probability >= 75:

            confidence = "HIGH"


        elif probability >= 55:

            confidence = "MEDIUM"


        else:

            confidence = "LOW"



        return {


            "probability":

                probability,


            "confidence":

                confidence,


            "trade_quality":

                (
                    "GOOD"
                    if probability >= 60
                    else
                    "AVOID"
                )

        }

In [14]:
prob_engine = ProbabilityEngine()


probability_result = prob_engine.calculate(

    trend="Bullish",

    pattern=pattern_result,

    price_action=price_action_result,

    current_price=2202,

    daily_support=levels["daily"]["nearest_support"],

    daily_resistance=levels["daily"]["nearest_resistance"],

    indicator_row=min5_df.iloc[-1],

    stop_loss=2145,

    target=2267

)


print(probability_result)

{'probability': 30, 'confidence': 'LOW', 'trade_quality': 'AVOID'}


Signal Engine - Done

In [15]:
# ==========================================================
# Signal Engine
#
# Combines:
#
# 1. Trend Engine
# 2. Chart Pattern Engine
# 3. Support Resistance Engine
# 4. Price Action Engine
# 5. Probability Engine
#
# Output:
# - LONG / SHORT / NO TRADE
# - Entry
# - Stop Loss
# - Target
# - Probability
# - Reason
# ==========================================================


class SignalEngine:


    def __init__(
            self,
            minimum_probability=60):

        self.minimum_probability = minimum_probability



    # ======================================================
    # ATR Based SL
    # ======================================================

    def _calculate_stop_loss(
            self,
            entry,
            atr,
            support=None):


        # Prefer support based SL

        if support:

            sl = support["price"] - (
                atr * 0.5
            )

        else:

            sl = entry - (
                atr * 1.5
            )


        return round(
            sl,
            2
        )



    # ======================================================
    # Target Calculation
    # ======================================================

    def _calculate_target(
            self,
            entry,
            resistance=None,
            atr=None):


        if resistance:

            target = resistance["price"]

        else:

            target = entry + (
                atr * 2
            )


        return round(
            target,
            2
        )



    # ======================================================
    # LONG Validation
    # ======================================================

    def _validate_long(
            self,
            trend,
            price_action,
            probability):


        if trend != "Bullish":

            return False


        if price_action["signal"] != "LONG":

            return False


        if (
            probability["probability"]
            <
            self.minimum_probability
        ):

            return False


        return True



    # ======================================================
    # Main Signal Generator
    # ======================================================

    def generate(

            self,

            symbol,

            trend_result,

            pattern_result,

            sr_result,

            price_action_result,

            probability_result,

            indicator_row):


        current_price = float(
            indicator_row["Close"]
        )


        atr = float(
            indicator_row["ATR"]
        )


        support = (
            sr_result["nearest_support"]
        )


        resistance = (
            sr_result["nearest_resistance"]
        )



        signal = "NO TRADE"

        entry = None

        stop_loss = None

        target = None



        # ==================================================
        # LONG Setup
        # ==================================================

        if self._validate_long(

            trend_result["Trend"],

            price_action_result,

            probability_result

        ):


            signal = "LONG"


            entry = round(
                current_price,
                2
            )


            stop_loss = self._calculate_stop_loss(

                entry,

                atr,

                support

            )


            target = self._calculate_target(

                entry,

                resistance,

                atr

            )



            # Safety Check

            if stop_loss >= entry:

                stop_loss = round(

                    entry -
                    (atr * 1.5),

                    2

                )



        # ==================================================
        # SHORT Setup
        # ==================================================

        elif (

            trend_result["Trend"]
            ==
            "Bearish"

            and

            price_action_result["signal"]
            ==
            "SHORT"

            and

            probability_result["probability"]
            >=
            self.minimum_probability

        ):


            signal = "SHORT"


            entry = round(
                current_price,
                2
            )


            stop_loss = round(

                entry +
                (atr * 1.5),

                2

            )


            if resistance:

                stop_loss = round(

                    resistance["price"]
                    +
                    (atr * 0.5),

                    2

                )


            target = round(

                entry -
                (atr * 2),

                2

            )



        # ==================================================
        # Final Response
        # ==================================================

        return {


            "symbol":

                symbol,


            "signal":

                signal,


            "entry":

                entry,


            "stop_loss":

                stop_loss,


            "target":

                target,


            "probability":

                probability_result["probability"],


            "confidence":

                probability_result["confidence"],


            "trend":

                trend_result["Trend"],


            "pattern":

                pattern_result["pattern"],


            "reason":

                [

                    "Trend confirmed",

                    "Price action validated",

                    "Support/Resistance checked",

                    "Probability filter passed"

                ]

                if signal != "NO TRADE"

                else

                [

                    "Conditions not satisfied"

                ]

        }

In [16]:
signal_engine = SignalEngine(
    minimum_probability=60
)


signal = signal_engine.generate(

    symbol="HINDUNILVR",

    trend_result=trend,

    pattern_result=pattern_result,

    sr_result=levels["daily"],

    price_action_result=price_action_result,

    probability_result=probability_result,

    indicator_row=min5_df.iloc[-1]

)


print("="*80)
print("FINAL TRADING SIGNAL for HINDUNILVR")
print("="*80)

for k,v in signal.items():
    print(k,":",v)

FINAL TRADING SIGNAL for HINDUNILVR
symbol : HINDUNILVR
signal : NO TRADE
entry : None
stop_loss : None
target : None
probability : 30
confidence : LOW
trend : Bearish
pattern : Double Top
reason : ['Conditions not satisfied']


In [17]:
# ==========================================================
# INITIALIZE ALL ENGINES
# ==========================================================

loader = DataLoader(
    "/home/hadoop/shareMarket_Data"
)


indicator_engine = IndicatorEngine()


trend_engine = TrendEngine()


sr_engine = SupportResistanceEngine()


pattern_engine = ChartPatternEngine()


price_action_engine = PriceActionEngine()


probability_engine = ProbabilityEngine()


signal_engine = SignalEngine(
    minimum_probability=60
)


print("All Engines Initialized")

All Engines Initialized


In [19]:
import time
from datetime import datetime


class Scheduler:


    def __init__(
            self,
            interval_minutes=5):

        self.interval = interval_minutes * 60

        self.running = False



    # ======================================================
    # Run Strategy Once
    # ======================================================

    def run_once(self):

        print("\n")
        print("=" * 80)
        print(
            "Strategy Running:",
            datetime.now()
        )
        print("=" * 80)


        # ---------------------------------
        # Load Latest Data
        # ---------------------------------

        symbols = [
            "HINDUNILVR",
            "BHEL",
            "TCS",
            "HCLTECH",
        ]


        for symbol in symbols:

            print("=" * 80)
            print("Processing :", symbol)
            print("=" * 80)
        
            daily_df, min5_df = loader.load_stock(symbol)
        
            daily_df = indicator_engine.calculate(daily_df)
            min5_df = indicator_engine.calculate(min5_df)
        
            trend_result = trend_engine.detect_trend(daily_df)
        
            sr_result = sr_engine.calculate(
                daily_df,
                min5_df
            )
        
            pattern_result = pattern_engine.calculate(daily_df)
        
            price_action_result = price_action_engine.calculate(
                min5_df,
                support=sr_result["daily"]["nearest_support"],
                resistance=sr_result["daily"]["nearest_resistance"]
            )
        
            current_price = float(min5_df.iloc[-1]["Close"])
            atr = float(min5_df.iloc[-1]["ATR"])
        
            support = sr_result["daily"]["nearest_support"]
            resistance = sr_result["daily"]["nearest_resistance"]
        
            stop_loss = (
                support["price"] - atr * 0.5
                if support else current_price - atr * 1.5
            )
        
            target = (
                resistance["price"]
                if resistance else current_price + atr * 2
            )
        
            probability_result = probability_engine.calculate(
                trend=trend_result["Trend"],
                pattern=pattern_result,
                price_action=price_action_result,
                current_price=current_price,
                daily_support=support,
                daily_resistance=resistance,
                indicator_row=min5_df.iloc[-1],
                stop_loss=stop_loss,
                target=target
            )
        
            signal = signal_engine.generate(
                symbol=symbol,
                trend_result=trend_result,
                pattern_result=pattern_result,
                sr_result=sr_result["daily"],
                price_action_result=price_action_result,
                probability_result=probability_result,
                indicator_row=min5_df.iloc[-1]
            )

            print("\n" + "=" * 100)
            print(f"SYMBOL        : {signal['symbol']}")
            print("=" * 100)
            
            print(f"Signal        : {signal['signal']}")
            print(f"Trend         : {signal['trend']}")
            print(f"Pattern       : {signal['pattern']}")
            
            print("-" * 100)
            
            print(f"Entry Price   : {signal['entry']}")
            print(f"Stop Loss     : {signal['stop_loss']}")
            print(f"Target        : {signal['target']}")
            
            print("-" * 100)
            
            print(f"Probability   : {signal['probability']} %")
            print(f"Confidence    : {signal['confidence']}")
            
            print("-" * 100)
            
            print("Reasons")
            
            for reason in signal["reason"]:
                print(f"  ✓ {reason}")
            
            print("=" * 100)



    # ======================================================
    # Start Scheduler
    # ======================================================

    def start(self):

        self.running = True


        print(
            "Scheduler Started..."
        )


        while self.running:


            now = datetime.now()


            # Run only on 5 minute candle close

            if (
                now.minute % 5 == 0
                and
                now.second <= 5
            ):

                self.run_once()


                time.sleep(
                    60
                )


            time.sleep(
                5
            )



    # ======================================================
    # Stop Scheduler
    # ======================================================

    def stop(self):

        self.running = False


        print(
            "Scheduler Stopped"
        )

In [ ]:
scheduler = Scheduler(
    interval_minutes=5
)

scheduler.start()

Scheduler Started...


Strategy Running: 2026-07-13 12:55:01.555383
Processing : HINDUNILVR

SYMBOL        : HINDUNILVR
Signal        : NO TRADE
Trend         : Bearish
Pattern       : Double Top
----------------------------------------------------------------------------------------------------
Entry Price   : None
Stop Loss     : None
Target        : None
----------------------------------------------------------------------------------------------------
Probability   : 30 %
Confidence    : LOW
----------------------------------------------------------------------------------------------------
Reasons
  ✓ Conditions not satisfied
Processing : BHEL

SYMBOL        : BHEL
Signal        : NO TRADE
Trend         : Bullish
Pattern       : Double Top
----------------------------------------------------------------------------------------------------
Entry Price   : None
Stop Loss     : None
Target        : None
--------------------------------------------------------------------------------

KeyboardInterrupt: 